# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

volatility_flag_success completed
vix_full_sample_lookahead completed


arima_forecast_block blocked
prompt_injection_unknown_capability blocked
spread_frequency_mismatch_block blocked


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

58 run records, 7 counted by the page, 609 events


total              58.000000
completed          23.000000
blocked            27.000000
failed              0.000000
rejected            1.000000
completed_share     0.396552
blocked_share       0.465517
failed_share        0.000000
rejected_share      0.017241
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

In [4]:
from quantlab.monitoring import (
    grounded_narratives,
    planner_refusals,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **planner_refusals(records),
    }
)

approved             20.000000
rejected              1.000000
approval_rate         0.952381
edit_rate             0.050000
approval_median_s     0.040500
approval_p95_s        8.422600
replays               4.000000
replay_match_rate     1.000000
narratives           23.000000
grounded_rate         1.000000
refused              18.000000
refused_share         0.310345
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,80
1,P01,PLANNER_REFUSED,BLOCK,18
2,V09,LOOK_AHEAD,APPROVE,13
3,V08,FREQUENCY_MISMATCH,BLOCK,7
4,V03,INVALID_PARAM,BLOCK,4
5,V02,UNKNOWN_CAPABILITY,BLOCK,2
6,E04,ALIGNMENT_LOSS,WARN,1
7,R01,REQUEST_TOO_LONG,BLOCK,0
8,V01,INVALID_PLAN,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,24
1,flag_regime,C07,22
2,resample,C02,15
3,compute_returns,C03,13
4,rolling_volatility,C06,13
5,forward_return,C08,13
6,compare_groups,C09,13
7,hac_mean_difference,C10,13
8,spread,C05,1
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,10,3066.8,4422.95,204.7,681.70,0.000409,0.000782,0.004090
1,narrator,10,617.5,1578.00,63.5,146.65,0.000094,0.000231,0.000935
2,total,10,3684.3,5652.90,268.2,804.95,0.000503,0.000968,0.005025


planner_calls             11.000000
cache_hits                 3.000000
hit_rate                   0.272727
saved_input_tokens     13140.000000
saved_output_tokens     2793.000000
saved_cost                 0.002711
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,58,629.056897,3507.6550,True
2,Validator,58,0.122759,0.3405,True
3,Executor,23,112.138261,159.4060,True
4,Narrator,23,680.978261,3080.5500,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-18T17:33:49.781Z,20261018-173349-1da385,blocked,Validator,V08,The plan has blocking issues: V08:s4
1,2026-10-18T17:33:49.745Z,20261018-173349-f841d5,blocked,Validator,V02,The plan has blocking issues: V02:s2
2,2026-10-18T17:33:49.707Z,20261018-173349-f22d5e,blocked,Planner,P01,The plan has blocking issues: P01:plan
3,2026-10-18T17:21:49.847Z,20261018-172149-67b4eb,blocked,Validator,V08,The plan has blocking issues: V08:s4
4,2026-10-18T17:21:49.816Z,20261018-172149-f7f9e0,blocked,Validator,V02,The plan has blocking issues: V02:s2
